# Notebook 3: Embeddings and Qdrant Indexing

## Purpose
Build dense and sparse indices for retrieval:
- Embed all chunks using OpenAI embeddings
- Create Qdrant vector collections
- Build BM25 sparse indices
- Validate index statistics and basic retrieval

## Outputs
- Qdrant collections (persisted locally)
- BM25 indices (JSON artifacts)
- Index summary statistics

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

import json
import pandas as pd
from collections import defaultdict

from rag.config import ARTIFACTS_DIR, get_default_pipeline_config, get_experiment_configs
from rag.pipeline import build_index

print(f'Project root: {project_root}')
print(f'Artifacts directory: {ARTIFACTS_DIR}')

In [ ]:
SMOKE_TEST = True

default_config = get_default_pipeline_config()
print(f'Default chunking strategy: {default_config["chunking_strategy"]}')
print(f'Default retrieval method: {default_config["retrieval_method"]}')
print(f'Embedding model: {default_config["embedding_model"]}')
print(f'Embedding dimensions: {default_config["embedding_dimensions"]}')

if SMOKE_TEST:
    print('\nSMOKE TEST MODE: Processing limited data')

## Build Index for Each Strategy

In [ ]:
print('Building indices...\n')

indices = {}

fixed_config = {**default_config, 'chunking_strategy': 'fixed_size'}
print('Building fixed-size index...')
try:
    vector_idx_fixed, bm25_idx_fixed, chunks_fixed = build_index(
        config=fixed_config,
        force_rebuild=False,
        companies=None
    )
    indices['fixed_size'] = {
        'vector_index': vector_idx_fixed,
        'bm25_index': bm25_idx_fixed,
        'chunks': chunks_fixed,
    }
    print(f'  Fixed-size index built: {len(chunks_fixed)} chunks')
except Exception as e:
    print(f'  Error building fixed-size index: {e}')

struct_config = {**default_config, 'chunking_strategy': 'structure_aware'}
print('\nBuilding structure-aware index...')
try:
    vector_idx_struct, bm25_idx_struct, chunks_struct = build_index(
        config=struct_config,
        force_rebuild=False,
        companies=None
    )
    indices['structure_aware'] = {
        'vector_index': vector_idx_struct,
        'bm25_index': bm25_idx_struct,
        'chunks': chunks_struct,
    }
    print(f'  Structure-aware index built: {len(chunks_struct)} chunks')
except Exception as e:
    print(f'  Error building structure-aware index: {e}')

print(f'\nSuccessfully built {len(indices)} indices')

## Index Statistics

In [ ]:
stats_rows = []

for strategy, idx_dict in indices.items():
    chunks = idx_dict['chunks']
    bm25 = idx_dict['bm25_index']
    
    vector_idx = idx_dict['vector_index']
    vector_count = len(chunks)
    bm25_doc_count = bm25.n_docs if hasattr(bm25, 'n_docs') else 0
    
    stats_rows.append({
        'Strategy': strategy,
        'Total Chunks': len(chunks),
        'Vector Embeddings': vector_count,
        'BM25 Docs': bm25_doc_count,
        'Avg Tokens': round(sum(c.token_count for c in chunks) / len(chunks), 1),
    })

stats_df = pd.DataFrame(stats_rows)
print('Index Statistics:')
print(stats_df.to_string(index=False))

## Quick Retrieval Sanity Check

In [ ]:
test_query = 'climate change mitigation and greenhouse gas emissions reduction'

for strategy, idx_dict in indices.items():
    print(f'\nStrategy: {strategy.upper()}')
    print(f'Test Query: {test_query}\n')
    
    bm25 = idx_dict['bm25_index']
    try:
        results = bm25.search(test_query, top_k=3)
        print('BM25 Results (Top 3):')
        for i, (score, chunk_id) in enumerate(results, 1):
            chunk = next((c for c in idx_dict['chunks'] if c.chunk_id == chunk_id), None)
            if chunk:
                page_info = f'p. {chunk.page_number}' if chunk.page_number >= 0 else 'unknown'
                preview = chunk.text[:100].replace('\\n', ' ') + '...'
                print(f'  {i}. [Score: {score:.3f}] {page_info} | {preview}')
    except Exception as e:
        print(f'  Error during BM25 retrieval: {e}')

## Embedding Space Validation

In [ ]:
import numpy as np
from rag.embedder import embed_query

test_query_emb = 'carbon emissions reporting'

for strategy, idx_dict in indices.items():
    print(f'\nEmbedding space validation for {strategy}:')
    
    chunks = idx_dict['chunks']
    vector_idx = idx_dict['vector_index']
    
    try:
        query_embedding = embed_query(test_query_emb)
        if query_embedding is None:
            print(f'  Skipping: Could not embed query')
            continue
        
        results = vector_idx.search(query_embedding, top_k=3)
        print(f'  Query: {test_query_emb}')
        print(f'  Top-3 dense results:')
        for i, (score, chunk_id) in enumerate(results, 1):
            chunk = next((c for c in chunks if c.chunk_id == chunk_id), None)
            if chunk:
                page_info = f'p. {chunk.page_number}' if chunk.page_number >= 0 else 'unknown'
                preview = chunk.text[:80].replace('\\n', ' ') + '...'
                print(f'    {i}. [Score: {score:.3f}] {page_info} | {preview}')
    except Exception as e:
        print(f'  Error during validation: {e}')

## Save Index Summary

In [ ]:
index_summary = {
    'timestamp': pd.Timestamp.now().isoformat(),
    'models': {
        'embedding_model': default_config['embedding_model'],
        'embedding_dimensions': default_config['embedding_dimensions'],
    },
    'indices': {}
}

for strategy, idx_dict in indices.items():
    chunks = idx_dict['chunks']
    bm25 = idx_dict['bm25_index']
    
    company_dist = defaultdict(int)
    for chunk in chunks:
        company_dist[chunk.company] += 1
    
    index_summary['indices'][strategy] = {
        'total_chunks': len(chunks),
        'total_tokens': sum(c.token_count for c in chunks),
        'avg_tokens_per_chunk': round(sum(c.token_count for c in chunks) / len(chunks), 1),
        'chunks_with_page_info': sum(1 for c in chunks if c.page_number >= 0),
        'chunks_with_tables': sum(1 for c in chunks if c.contains_table),
        'bm25_indexed_docs': bm25.n_docs if hasattr(bm25, 'n_docs') else 0,
        'company_distribution': dict(company_dist),
    }

summary_path = ARTIFACTS_DIR / 'index_summary.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(index_summary, f, indent=2)

print(f'Index summary saved to: {summary_path}')
print('\nIndex Summary (truncated):')
print(json.dumps(index_summary, indent=2)[:500] + '...')